# YES Bank stock-price model — Google Colab

This notebook downloads the project from GitHub, installs its pinned packages, trains and evaluates the models, and displays the results. The supplied monthly CSV ends in November 2020, so its forecast is historical (December 2020), not a current market estimate. This project is educational and is not investment advice.

## 1. Get the project

Run this cell first. If you uploaded an updated CSV, set `DATA_PATH` to its Colab path in the optional cell below.

In [ ]:
from pathlib import Path
import subprocess

PROJECT = Path('/content/yes-bank-machine-learning-model-')
REPO_URL = 'https://github.com/harshu1011/yes-bank-machine-learning-model-.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)
DATA_PATH = PROJECT / 'data' / 'data_YesBank_StockPrices.csv'
print('Project:', PROJECT)
print('Data:', DATA_PATH)

## Optional: use a different CSV

Skip this cell to use the dataset already in the repository. To use your own monthly OHLC CSV, run this cell and choose the file. It must have `Date`, `Open`, `High`, `Low`, and `Close` columns, with dates formatted like `Jan-20`.

In [ ]:
# Uncomment and run to upload your own CSV instead of using the repository dataset.
# from google.colab import files
# uploaded = files.upload()
# if uploaded:
#     DATA_PATH = Path('/content') / next(iter(uploaded))
#     print('Using uploaded data:', DATA_PATH)

## 2. Install the project packages

The project pins package versions so the saved model and training workflow use consistent libraries.

In [ ]:
%cd /content/yes-bank-machine-learning-model-
%pip install -r requirements.txt

## 3. Train and evaluate

The program compares Ridge, Random Forest, and Histogram Gradient Boosting with a persistence baseline. It uses chronological time-series folds for model selection and keeps the final 20% of observations for holdout evaluation. An ML model is selected only if its cross-validation MAE improves on persistence by at least 5%; otherwise, the persistence forecast is selected.

In [ ]:
import subprocess
subprocess.run(['python', 'train.py', '--data', str(DATA_PATH)], check=True)

## 4. View the evaluation results

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

metrics = json.loads((PROJECT / 'artifacts' / 'metrics.json').read_text())
print('Selected forecast:', metrics['selection']['selected_model'])
print('Best ML candidate:', metrics['selection']['best_ml_candidate'])
print('Cross-validation improvement over persistence: {:.2f}%'.format(100 * metrics['selection']['cv_improvement_fraction']))
print('Holdout period:', metrics['split']['test_first_target_month'], 'to', metrics['split']['test_last_target_month'])
comparison = pd.DataFrame({name: values for name, values in metrics['holdout_metrics'].items()}).T
display(comparison.round(2))
display(Image(filename=str(PROJECT / 'artifacts' / 'holdout_mae_comparison.png')))
display(Image(filename=str(PROJECT / 'artifacts' / 'actual_vs_predicted.png')))
display(Image(filename=str(PROJECT / 'artifacts' / 'residuals.png')))

## 5. Forecast the month after the latest row

For the supplied dataset this will display December 2020. To get a forecast for a newer period, upload a newer CSV in the optional section above.

In [ ]:
subprocess.run(['python', 'predict.py', '--data', str(DATA_PATH)], check=True)
predictions = pd.read_csv(PROJECT / 'artifacts' / 'test_predictions.csv')
display(predictions.tail(10))